# 17c · Why did variant A fail on "A Cowboy Who Rode the Waves"? (analysis only)

On the Cowboy prompt, the leave-one-prompt-out model of variant A (spelling/typo augmentation) flagged **17.42%** of the human essays at its 1% threshold, against **4.30%** for the original recipe (`17b_augment_results.ipynb`, section 4). No model is trained or rerun here; the saved fold scores and the augmentation cache are enough. Questions:

1. Do Cowboy human essays have an unusual share of tokens outside the GloVe vocabulary (the measure the augmentation was built on)?
2. How do the essays that variant A newly flags differ from the ones it does not?
3. What did the spelling corrector do to proper nouns and rare words in the human training essays?
4. What do ten essays flagged by A but not by the original look like, read closely?

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')
import re
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats

from src.data import load_splits
from src.evaluate import threshold_for_fpr, ClusterSampler, bootstrap_interval
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 110)
SEED = 42
COWBOY = '"A Cowboy Who Rode the Waves"'
data = load_splits()
he = pd.read_csv('../results/16b_lopo_human_essays.csv', index_col='row_id').join(data[['dup_cluster', 'source']])   # every human essay of the 13 prompts, with features
print(he.shape, '| Cowboy humans:', (he['prompt_name'] == COWBOY).sum())

(22907, 12) | Cowboy humans: 1372


## 1 · Out-of-vocabulary share by prompt

`misspell_pct` is the share of an essay's tokens missing from the 400,000-word GloVe vocabulary (the measure of notebook 14, and the test the corrector uses). It counts misspellings, but also names, rare words and slang.

In [2]:
g = he.groupby('prompt_name')['misspell_pct'].agg(['mean', 'median', lambda s: (s > 2).mean() * 100]).rename(columns={'<lambda_0>': '% essays above 2%'})
g['rank (1 = highest mean)'] = g['mean'].rank(ascending=False).astype(int)
print(g.sort_values('mean', ascending=False).round(2).to_string())
c, o = he.loc[he['prompt_name'] == COWBOY, 'misspell_pct'], he.loc[he['prompt_name'] != COWBOY, 'misspell_pct']
print(f"\nCowboy mean {c.mean():.2f}% vs other prompts {o.mean():.2f}%  | Mann-Whitney p = {stats.mannwhitneyu(c, o).pvalue:.3g}")

                                       mean  median  % essays above 2%  rank (1 = highest mean)
prompt_name                                                                                    
Exploring Venus                        2.04    1.57              37.33                        1
Facial action coding system            1.89    1.45              35.63                        2
"A Cowboy Who Rode the Waves"          1.65    1.24              29.81                        3
Car-free cities                        1.59    1.14              25.96                        4
Driverless cars                        1.52    1.15              25.77                        5
The Face on Mars                       1.48    1.08              25.77                        6
Distance learning                      0.53    0.00               8.16                        7
Mandatory extracurricular activities   0.45    0.00               6.23                        8
Seeking multiple opinions              0

## 2 · Newly flagged versus not (Cowboy humans, fold 0)

Each model's threshold is the score that flags 1% of the val humans of the other 12 prompts (as everywhere). Groups: flagged by A only ("newly flagged"), by both, by neither.

In [3]:
def fold0(path):
    s = pd.read_csv(path, index_col='row_id').sort_index()
    vh = s[(s['role'] == 'other_val') & (s['label'] == 0)]['logit']
    lh = s[(s['role'] == 'left_out') & (s['label'] == 0)]['logit']
    return lh, threshold_for_fpr(vh, 0.01), vh
lo, t_o, vh_o = fold0('../results/16b_lopo/scores_fold00_seed42.csv')
la, t_a, vh_a = fold0('../results/17_variants/A/scores_fold00_seed42.csv')
assert lo.index.equals(la.index)
grp = np.where((la >= t_a) & (lo < t_o), 'A only (newly flagged)', np.where((la >= t_a) & (lo >= t_o), 'both', np.where((la < t_a) & (lo >= t_o), 'original only', 'neither')))
cw = he.loc[lo.index].assign(group=grp, logit_orig=lo, logit_A=la)
cw['shift'] = cw['logit_A'] - cw['logit_orig']
print('thresholds: original', round(t_o, 2), '| A', round(t_a, 2))
print(cw['group'].value_counts().to_string())
FEAT = ['n_words', 'misspell_pct', 'log_ppl', 'ai_vocab_pct', 'logit_orig', 'logit_A', 'shift']
print(cw.groupby('group')[FEAT].median().round(2).to_string())
print('\nsource of the newly flagged essays:', cw.loc[cw['group'] == 'A only (newly flagged)', 'source'].value_counts().to_dict())
new, rest = cw[cw['group'] == 'A only (newly flagged)'], cw[cw['group'] == 'neither']
for f in ['misspell_pct', 'n_words', 'log_ppl', 'ai_vocab_pct']:
    print(f"{f:14s} newly flagged median {new[f].median():6.2f} vs neither {rest[f].median():6.2f}   Mann-Whitney p = {stats.mannwhitneyu(new[f], rest[f]).pvalue:.2g}")

thresholds: original 6.94 | A 6.4
group
neither                   1133
A only (newly flagged)     180
both                        59
                        n_words  misspell_pct  log_ppl  ai_vocab_pct  logit_orig  logit_A  shift
group                                                                                           
A only (newly flagged)    267.5          0.95     3.21           0.0       -2.59     7.85  10.21
both                      269.0          0.52     3.06           0.0        7.86     8.81   0.68
neither                   265.0          1.36     3.52           0.0       -8.03    -6.62   1.22

source of the newly flagged essays: {'persuade_corpus': 180}
misspell_pct   newly flagged median   0.95 vs neither   1.36   Mann-Whitney p = 2.8e-06
n_words        newly flagged median 267.50 vs neither 265.00   Mann-Whitney p = 0.67
log_ppl        newly flagged median   3.21 vs neither   3.52   Mann-Whitney p = 1.1e-16
ai_vocab_pct   newly flagged median   0.00 vs neither   0.0

In [4]:
# is the shift (A minus original logit) related to the essay's misspelling rate, inside Cowboy?
print('Spearman of the A-minus-original logit shift with misspelling %:', round(stats.spearmanr(cw['shift'], cw['misspell_pct'])[0], 3))
print('Spearman of the shift with the original model logit:', round(stats.spearmanr(cw['shift'], cw['logit_orig'])[0], 3))
cw['misspell band'] = pd.cut(cw['misspell_pct'], [-0.01, 0, 1, 2, 100], labels=['0%', '0-1%', '1-2%', '>2%'])
t = cw.groupby('misspell band', observed=True).agg(n=('shift', 'size'), flagged_orig=('logit_orig', lambda s: (s >= t_o).mean() * 100), flagged_A=('logit_A', lambda s: (s >= t_a).mean() * 100), median_shift=('shift', 'median'))
t.round(2)

Spearman of the A-minus-original logit shift with misspelling %: 0.075
Spearman of the shift with the original model logit: 0.48


,n,flagged_orig,flagged_A,median_shift
misspell band,,,,
0%,128,13.28,33.59,1.10
0-1%,436,6.88,22.48,1.36
1-2%,399,2.26,16.54,1.64
>2%,409,0.73,7.82,1.54


## 3 · What the corrector did to the human training essays

The 9,065 human training essays whose spelling was repaired (`data/external/17_augmented_train.csv`). The corrector only changes single words (a repaired essay has the same number of whitespace-separated tokens), so each changed token can be listed.

In [5]:
aug = pd.read_csv('../data/external/17_augmented_train.csv', index_col='row_id')
tr = data[(data['split'] == 'train') & (data['label'] == 0)]
fixed = aug.loc[tr.index]; fixed = fixed[fixed['kind'] == 'spell_fixed']
pairs, n_essays = [], 0
for i in fixed.index:
    a, b = tr.loc[i, 'text'].split(), fixed.loc[i, 'text_aug'].split()
    if len(a) == len(b):
        n_essays += 1
        pairs += [(x, y, i) for x, y in zip(a, b) if x != y]
P = pd.DataFrame(pairs, columns=['before', 'after', 'row_id'])
strip = lambda w: re.sub(r"[^A-Za-z']", '', w)
P['b'], P['a'] = P['before'].map(strip), P['after'].map(strip)
P = P[(P['b'] != '') & (P['b'].str.lower() != P['a'].str.lower())]
P['contraction'] = P['a'].str.contains("'") & ~P['b'].str.contains("'")
P['cap_before'] = P['b'].str[0].str.isupper()
P['long'] = P['b'].str.len() >= 9
def lev(x, y):
    prev = list(range(len(y) + 1))
    for i, cx in enumerate(x, 1):
        cur = [i]
        for j, cy in enumerate(y, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (cx != cy)))
        prev = cur
    return prev[-1]
samp = P.sample(min(8000, len(P)), random_state=SEED)
P.loc[samp.index, 'dist'] = [lev(x.lower(), y.lower()) for x, y in zip(samp['b'], samp['a'])]
print(f"{n_essays:,} repaired essays, {len(P):,} word changes ({len(P) / n_essays:.1f} per essay)")
print(f"contractions mapped (dont -> don't ...): {P['contraction'].mean() * 100:.1f}% | begin with a capital letter: {P['cap_before'].mean() * 100:.1f}% | 9+ letters: {P['long'].mean() * 100:.1f}% | "
      f"edit distance 1: {(P['dist'] == 1).mean() / P['dist'].notna().mean() * 100:.0f}%, distance 2+: {(P['dist'] >= 2).sum() / P['dist'].notna().sum() * 100:.0f}% of a {int(P['dist'].notna().sum()):,} sample")
top = Counter(zip(P['b'].str.lower(), P['a'].str.lower())).most_common(30)
print('\nthirty most frequent changes:', ', '.join(f'{b}->{a} ({n})' for (b, a), n in top))

9,065 repaired essays, 34,173 word changes (3.8 per essay)
contractions mapped (dont -> don't ...): 18.9% | begin with a capital letter: 4.0% | 9+ letters: 33.7% | edit distance 1: 80%, distance 2+: 20% of a 8,000 sample

thirty most frequent changes: dont->don't (1862), thats->that's (801), cant->can't (765), wont->won't (681), driveless->driverless (520), im->i'm (352), wouldnt->wouldn't (281), theres->there's (252), didnt->didn't (245), doesnt->doesn't (225), whats->what's (214), polution->pollution (213), isnt->isn't (172), diffrent->different (122), arent->aren't (116), shouldnt->shouldn't (105), benifit->benefit (98), enviorment->enviroment (97), enviornment->environment (91), wasnt->wasn't (84), somthing->something (67), tecnology->technology (64), completly->completely (61), oppurtunity->opportunity (60), couldnt->couldn't (60), incase->incas (56), emmisions->emmissions (55), paragragh->paragraph (54), benifits->benefits (52), useing->using (48)


In [6]:
# changes most likely to be wrong: words that start with a capital (names at the start of a sentence) or are 9+ letters long; 50 at random, to be read
risky = P[P['cap_before'] | P['long']].drop_duplicates(['b', 'a'])
print(len(risky), 'distinct risky changes; 50 random ones:')
print(', '.join(f"{r.b}->{r.a}" for r in risky.sample(50, random_state=SEED).itertuples()))

6730 distinct risky changes; 50 random ones:
minutesto->minutest, Accoding->According, frustratation->frustration, temperatues->temperatures, civilizationon->civilization, entertaning->entertaining, emissionsin->emissions, hazordous->hazardous, expresssions->expressions, tempperatures->temperatures, Unfortunatlly->Unfortunately, glithched->glitches, techniacally->technically, Vistiting->Visiting, develloped->developed, caputured->captured, innapropriate->inappropriate, voilations->violations, convienience->convenience, constucts->constructs, perperson->peterson, comfirmed->confirmed, illastrate->illustrate, diffculities->difficulties, classroms->classrooms, consistet->consistent, sociologits->sociologist, councelers->counselors, speacialy->special, seightseeing->sightseeing, charcteristic->characteristic, scienetists->scientists, explaitory->explanatory, extratrrestrial->extraterrestrial, protectionn->protection, mounument->monument, preferces->preferred, manfacturers->manufacturers, s

## 4 · Ten humans flagged by variant A but not by the original, read closely

In [7]:
ten = new.sample(10, random_state=SEED)
for rid, r in ten.iterrows():
    print(f"[row {rid} | {int(r['n_words'])} words | misspell {r['misspell_pct']:.1f}% | log-ppl {r['log_ppl']:.2f} | AI-vocab {r['ai_vocab_pct']:.1f}% | logit orig {r['logit_orig']:+.1f} -> A {r['logit_A']:+.1f}]")
    print('   ', data.loc[rid, 'text'][:650].replace('\n', ' '), '...\n')

[row 5015 | 794 words | misspell 0.4% | log-ppl 2.43 | AI-vocab 0.1% | logit orig +0.3 -> A +6.4]
    Boy am I glad there's no more homework!  Hi, my name is Luke, I just finished high school and want to become a doctor.  I love the idea of helping people.  That's why I'm signing up to become a "Seagoing Cowboy."  A Seagoing Cowboy is a person that goes on a boat across all kinds of different oceans to help countries that were effected by World War Two.  Also, I would like to become a Seagoing Cowboy because I would like to be remebered in different countries for heroic actions.  Not very many people can be famous for their actions in different places because there's no way to spread the news.  The only people that really remember you are the ...

[row 5208 | 280 words | misspell 1.8% | log-ppl 3.72 | AI-vocab 0.0% | logit orig -6.3 -> A +8.5]
    Hi, i'm Luke, and one day I was working at the bank and my friend, Don ask "Hey Luke, you want to go to Europe on a cattle boat with me!" Ho

## 5 · Regression to the mean? The 13 prompts together

The seven pre-registered folds were chosen from the extremes of the original recipe's false-flag rates. If part of the Phase 3 gain were regression to the mean, the change (A minus original) would be more negative where the original's rate was high. Across the 13 prompts:

In [8]:
byp = pd.read_csv('../results/17b_variantA_13fold_by_prompt.csv', index_col='prompt')
x, y = byp['baseline %'], byp['difference (pts)']
rho, p = stats.spearmanr(x, y)
rho2, p2 = stats.spearmanr(x.drop(COWBOY), y.drop(COWBOY))
print(f"Spearman correlation of (A minus original) with the original's false-flag rate: all 13 prompts rho = {rho:.2f} (p = {p:.2f}); without Cowboy rho = {rho2:.2f} (p = {p2:.3f})")
print('by original rate, the three highest prompts:', byp.sort_values('baseline %', ascending=False).head(3)['difference (pts)'].round(2).to_dict())
print('mean change, 7 highest-rate prompts vs 6 lowest-rate prompts:', round(byp.sort_values('baseline %', ascending=False).head(7)['difference (pts)'].mean(), 2), 'vs', round(byp.sort_values('baseline %', ascending=False).tail(6)['difference (pts)'].mean(), 2))
byp[['baseline %', 'variant %', 'difference (pts)']].sort_values('baseline %', ascending=False).round(2)

Spearman correlation of (A minus original) with the original's false-flag rate: all 13 prompts rho = -0.34 (p = 0.26); without Cowboy rho = -0.47 (p = 0.124)
by original rate, the three highest prompts: {'Seeking multiple opinions': -0.45, 'Distance learning': -2.6, 'Community service': -2.2}
mean change, 7 highest-rate prompts vs 6 lowest-rate prompts: 0.88 vs 0.2


,baseline %,variant %,difference (pts)
prompt,,,
Seeking multiple opinions,5.80,5.35,-0.45
Distance learning,5.52,2.92,-2.60
Community service,5.19,2.98,-2.20
Exploring Venus,4.73,5.32,0.59
"""A Cowboy Who Rode the Waves""",4.30,17.42,13.12
Mandatory extracurricular activities,3.47,2.22,-1.26
Facial action coding system,2.54,1.52,-1.01
Grades for extracurricular activities,1.60,2.40,0.80
Phones and driving,1.54,1.71,0.17


In [9]:
summary = {'cowboy_misspell_mean': float(c.mean()), 'others_misspell_mean': float(o.mean()), 'cowboy_misspell_rank_of_13': int(g.loc[COWBOY, 'rank (1 = highest mean)']),
           'newly_flagged_n': int((cw['group'] == 'A only (newly flagged)').sum()), 'both_n': int((cw['group'] == 'both').sum()), 'neither_n': int((cw['group'] == 'neither').sum()),
           'orig_flagged_misspell_gt2': float(t.loc['>2%', 'flagged_orig']), 'A_flagged_misspell_gt2': float(t.loc['>2%', 'flagged_A']), 'n_misspell_gt2': int(t.loc['>2%', 'n']),
           'flagged_misspell_gt2_orig_k': int(round(t.loc['>2%', 'flagged_orig'] * t.loc['>2%', 'n'] / 100)), 'flagged_misspell_gt2_A_k': int(round(t.loc['>2%', 'flagged_A'] * t.loc['>2%', 'n'] / 100)),
           'corrector_changes': int(len(P)), 'corrector_contractions': int(P['contraction'].sum()), 'corrector_cap_initial': int(P['cap_before'].sum()),
           'spearman_diff_vs_orig_rate_all13': float(rho), 'spearman_p_all13': float(p), 'spearman_excl_cowboy': float(rho2), 'spearman_p_excl_cowboy': float(p2)}
pd.Series(summary).to_csv('../results/17c_cowboy_summary.csv', header=['value'])
pd.Series(summary)

cowboy_misspell_mean                    1.645393
others_misspell_mean                    1.013802
cowboy_misspell_rank_of_13              3.000000
newly_flagged_n                       180.000000
both_n                                 59.000000
neither_n                            1133.000000
orig_flagged_misspell_gt2               0.733496
A_flagged_misspell_gt2                  7.823961
n_misspell_gt2                        409.000000
flagged_misspell_gt2_orig_k             3.000000
flagged_misspell_gt2_A_k               32.000000
corrector_changes                   34173.000000
corrector_contractions               6451.000000
corrector_cap_initial                1368.000000
spearman_diff_vs_orig_rate_all13       -0.335165
spearman_p_all13                        0.262943
spearman_excl_cowboy                   -0.468531
spearman_p_excl_cowboy                  0.124455
dtype: float64

## Findings (a short paragraph, after reading the outputs above)

**Cowboy is not special for its vocabulary, but for its style.** Its share of tokens outside GloVe is 1.65% against 1.01% for the other prompts, which puts it third of 13: the six source-based prompts (Exploring Venus, Facial action coding system, Cowboy, Car-free cities, Driverless cars, The Face on Mars) all sit at 1.5-2.0%, and Exploring Venus (2.04%) moved only +0.59 points under variant A, so the share of out-of-vocabulary tokens cannot be the whole story. The 180 humans that variant A newly flags (of 1,372) are no longer than the rest (267 vs 265 words) but are **more predictable** (GPT-2 log-perplexity 3.21 vs 3.52) and slightly cleaner (0.95% vs 1.36%); variant A moved them by a median of **+10.2 logits** (from -2.6, clearly human to the original, to +7.9). They are formulaic first-person persuasive letters written as a "Seagoing Cowboy" (all 180 from the Persuade corpus), and of the ten read closely **seven contain obvious typos** ("shiped", "experince", "majic", "frineds", "Definetely", "oppurtunity", "thta") that the original model treated as human evidence: one essay with a 1.8% misspelling rate went from -6.3 (human) to +8.5 (flagged). That is what variant A was built to do. Injecting typos into half of the AI training essays and repairing half of the human ones makes misspellings uninformative, and for a prompt whose human essays are formulaic and persuasive, the typos were most of what marked them as human: in the Cowboy essays with more than 2% misspellings, the original flagged 0.7% and A flags 7.8%, and the flag rate rose in every misspelling band. The ten essays are consistent with that reading, which is a mechanism that fits, not one this notebook isolates (no ablation was run).

**The corrector.** It changed 3.8 words per repaired essay (34,173 changes in 9,065 essays); **19%** of the changes map informal contractions (dont, thats, cant, wont, im ... to their apostrophe forms), which removes another informal marker of student writing, and the most frequent change after those is a topical misspelling ("driveless" to "driverless", 520 times). Proper nouns were rarely touched (4% of changes start with a capital letter, and mid-sentence capitals are skipped by design), so damage to names is limited; the errors are mostly rare words repaired to the wrong word, and in a few frequent cases the "repair" is itself still a non-word ("incase" to "incas", 56 times; "emmisions" to "emmissions", 55 times; "enviorment" to "enviroment", 97 times). In 50 random risky repairs, about one in six was wrong ("perperson" to "peterson", "shappiness" to "sappiness"), matching the one-in-six found in the earlier sample.

**Regression to the mean.** The change under variant A is more negative where the original's rate was higher, as regression to the mean (or a real effect on high-rate prompts) would predict, but weakly: Spearman rho = -0.34 across the 13 prompts (p = 0.26), -0.47 without Cowboy (p = 0.12). Without Cowboy, the six highest-rate prompts improved by 1.15 points on average and the lowest-rate ones did not (+0.2). **This is suggestive, not established**: seed noise measured on one fold was 0.34 points, and the only direct test, rerunning the original recipe with other seeds on an extreme-rate prompt (fold 11, Seeking multiple opinions, original rate 5.80%), is part of Phase 5. The likely reading is therefore: a real gain on some high-rate prompts, plus selection of those prompts for being extreme, plus a real failure on a prompt where typos were the human cue.